In [1]:
#install modules and dependencies

import sys

# Force install directly in Kernel
!{sys.executable} -m pip install -U langchain langchain-openai langchain-anthropic deepagents langchain-openrouter


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
#build a basic agent

MY_KEY ="my key"

#get API Key from AI model provider (OpenRouter)
import os
os.environ["OPENAI_API_KEY"] = MY_KEY
os.environ["OPENROUTER_API_KEY"] = MY_KEY

os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"


#build a basic agent (chosen language model : openrouter:openrouter/free, tool : basic weather function, prompt : simple)
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openrouter:openrouter/free",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'reasoning', 'reasoning': "Okay, the user asked about the weather in San Francisco. I called the get_weather function with the city as San Francisco. The response came back saying it's always sunny. Now I need to relay that information to the user in a friendly manner. Let me make sure to present the response clearly. Maybe add a smiley to keep it light. Check if there's any additional info needed, but since the user just asked for the weather, the response should suffice. Alright, time to put it all together.\n"}, {'type': 'text', 'text': '\n\nThe weather inSan Francisco is always sunny! ☀️\n'}]


In [ ]:
#build a real-world agent

#1. Define the system prompt
SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

#2. Create tools
import urllib.error
import urllib.request

from langchain.tools import tool

@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL.
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text

#3. Configure model
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openrouter:openrouter/free",
    temperature=0.5,
    timeout=300,
    max_tokens=25000,
)

#4. Add memory
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

#(7. Debug, get trace with LangSmith) 
MY_KEY ="my key"

import os
os.environ["LANGSMITH_API_KEY"] = MY_KEY
os.environ["LANGSMITH_TRACING"] = "true"

#5. Create and run agents (Deep agent and LangChain agent)
from langchain.agents import create_agent
from deepagents import create_deep_agent

agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

deep_agent = create_deep_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

#content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
#URL: https://www.gutenberg.org/files/64317/64317-0.txt

#Answer as much as you can:

#1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
#2) The 1-based line number of the first line in the file that contains `Daisy`.
#3) A two-sentence neutral synopsis.

#Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
#your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
#the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

#agent_result = agent.invoke(
#    {"messages": [{"role": "user", "content": content}]},
#    config={"configurable": {"thread_id": "great-gatsby-lc"}},
#)
#deep_agent_result = deep_agent.invoke(
#    {"messages": [{"role": "user", "content": content}]},
#    config={"configurable": {"thread_id": "great-gatsby-da"}},
#)
#print(agent_result["messages"][-1].content_blocks)
#print("\n")
#print(deep_agent_result["messages"][-1].content_blocks)

# Un texte de test court avec une structure identique pour valider la logique
content = """We are testing the agent with a short stable document.
URL: https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.5/README.md

Answer the following questions based on the document text:
1) What is the name of the tool discussed in this README?
2) What is its main purpose in one short sentence?
"""

print("Calling agent...")
agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "test-github-url"}},
)
print("Agent finished!")

# Affichage propre du résultat
try:
    if hasattr(agent_result["messages"][-1], "content_blocks") and agent_result["messages"][-1].content_blocks:
        for block in agent_result["messages"][-1].content_blocks:
            if isinstance(block, dict) and block.get("type") == "text":
                print("\n--- Response ---")
                print(block["text"])
            elif hasattr(block, "type") and block.type == "text":
                print("\n--- Response ---")
                print(block.text)
    else:
        print("\n--- Response ---")
        print(agent_result["messages"][-1].content)
except Exception:
    print("\n--- Response ---")
    print(agent_result["messages"][-1])


#6. Review the results

#7. Debug (trace agent calls with LangSmith)
#See above model creation